## Executive Summary

A logistics operations team plans a randomized field trial comparing three driver-routing strategies (static baseline, dynamic rerouting, AI-optimized) across three depot regions, with average delivery delay (minutes) as the response. PROC GLMPOWER takes an *exemplary* dataset of conjectured cell means and solves for the total number of drivers needed to detect the headline AI-Optimized-vs-Static improvement at 80% and 90% power, then maps how achievable power erodes as route-to-route variability grows.

# Sizing a Driver-Routing Field Trial with PROC GLMPOWER

## Executive Summary

A logistics operations team is about to launch a randomized field trial comparing three driver-routing strategies -- a **Static** baseline, a **Dynamic** rerouting system, and an **AI-Optimized** planner -- deployed across three depot regions (Northeast, Midwest, West). The response is average **delivery delay in minutes**. Before committing fleet capacity to the trial, the team must answer: *how many drivers do we need to reliably detect the operational improvement we expect?*

This notebook uses **PROC GLMPOWER** to perform prospective power and sample-size analysis for the general linear model behind the trial. Working from an *exemplary* dataset of conjectured cell means, it (1) solves for the total enrollment needed to reach 80% and 90% power for the headline planned contrast -- **AI-Optimized vs. Static** delay -- under the two-way model, (2) maps how achievable power for the omnibus strategy effect degrades as route-to-route variability rises, and (3) traces a power curve to support the enrollment decision.

> **Key takeaway:** Under a plausible error standard deviation of 8 minutes, **63 drivers** deliver at least 80% power (achieved **0.8035**) and **83 drivers** deliver at least 90% power (achieved **0.9014**) for detecting the AI-Optimized-vs-Static delay reduction -- but if delay variability runs closer to 10 minutes, even 90 enrolled drivers leave the broader strategy comparison short of 70% power (**0.6782**), underscoring the value of tight, well-instrumented routes.

---
## 1. Build the Exemplary Design

The first step encodes the trial layout and the team's *conjectured* mean delay for every routing-strategy x depot-region combination. We fix a random seed and add a negligible jitter so the means look realistic while the 3x3 balanced structure is preserved. The `cell_n` weight of 1 in every cell tells GLMPOWER the design is balanced.

In [1]:
/* ================================================================
   Generate the exemplary dataset of conjectured mean delays.
   One row per routing-strategy x depot-region design cell.
   ================================================================ */
data routing_trial;
   call streaminit(20260531);
   length routing_strategy $8 depot_region $9;
   array strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   array region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   array smean[3]     _temporary_ (18.0 14.5 11.0);   /* conjectured strategy means */
   array radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regional adjustments (min)  */
   do i = 1 to 3;
      do j = 1 to 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         output;
      end;
   end;
   drop i j;
run;

proc print data=routing_trial label noobs;
   var routing_strategy depot_region mean_delay_min cell_n;
   title "Exemplary Cell Means: Conjectured Delivery Delay (minutes)";
run;

                               Exemplary Cell Means: Conjectured Delivery Delay (minutes)                               

routing_strategy  depot_region  mean_delay_min  cell_n
Static            Northeast       19.687507651       1
Static            Midwest        17.9871067398       1
Static            West           16.8240210086       1
Dynamic           Northeast      15.9537535365       1
Dynamic           Midwest        14.4415169858       1
Dynamic           West           12.8636389493       1
AIOpt             Northeast      12.6143811724       1
AIOpt             Midwest         10.893885771       1
AIOpt             West             9.635351403       1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Sample Size for the Headline Planned Contrast

With the design in hand, we ask GLMPOWER to **solve for total sample size** (`NTOTAL = .`) at 80% and 90% power. The `MODEL` statement specifies the full two-way layout with interaction (`routing_strategy*depot_region`), `WEIGHT cell_n` declares the balanced profile weights, and `STDDEV = 8` is the assumed root mean square error of delivery delay.

We pre-register three planned `CONTRAST` comparisons -- AI-Opt vs. Static, Dynamic vs. Static, and any-reroute vs. Static -- which document the operationally meaningful, single-degree-of-freedom hypotheses the trial is built to test. The **Computed N Total** table below sizes the trial for the headline **AI-Opt vs. Static** contrast (the primary comparison), reporting the smallest enrollment that meets each power target.

`NFRACTIONAL` controls how that enrollment is rounded: GLMPOWER finds the *exact* (fractional) total needed to hit the nominal power, then reports the next whole driver up (the **ceiling**) as `N Total`. Because the rounded-up trial is slightly larger than strictly required, the realized power at that integer enrollment -- shown in the **Actual Power** column -- sits just *above* the nominal target.

In [2]:
/* ================================================================
   Solve for the total drivers needed to reach 80% and 90% power
   for the headline AI-Opt vs. Static contrast on delivery delay.
   ================================================================ */
proc glmpower data=routing_trial;
   class routing_strategy depot_region;
   model mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   weight cell_n;
   contrast "AI-Opt vs. Static"      routing_strategy -1  0  1;
   contrast "Dynamic vs. Static"     routing_strategy -1  1  0;
   contrast "Any reroute vs. Static" routing_strategy -2  1  1;
   power
      nfractional
      stddev = 8.0
      alpha  = 0.05
      ntotal = .
      power  = 0.80 0.90;
   title "Sample Size to Detect the AI-Opt vs. Static Delay Contrast";
run;

                               Exemplary Cell Means: Conjectured Delivery Delay (minutes)                               

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  mean_delay_min               
Source              ROUTING_STRATEGY             
Source              DEPOT_REGION                 
Source              ROUTING_STRATEGY*DEPOT_REGION

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Power Sensitivity to Variability and Trial Size

The sample-size answer hinges on the assumed error standard deviation, which is rarely known precisely in advance. Here we flip the question: dropping the (negligible) interaction to focus on the **omnibus strategy effect**, we **fix** several candidate enrollment totals (`NTOTAL = 45 90 180`) and **solve for achieved power** (`POWER = .`) across a grid of plausible delay standard deviations (6, 8, and 10 minutes). The resulting table is a sensitivity map -- it shows how robust each enrollment plan is for detecting *any* strategy difference if real-world route variability turns out higher than hoped.

In [3]:
/* ================================================================
   Sensitivity grid: achieved power across candidate trial sizes
   and plausible delay standard deviations.
   ================================================================ */
proc glmpower data=routing_trial;
   class routing_strategy depot_region;
   model mean_delay_min = routing_strategy depot_region;
   weight cell_n;
   power
      nfractional
      stddev = 6.0 8.0 10.0
      alpha  = 0.05
      ntotal = 45 90 180
      power  = .;
   title "Achieved Power Across Variability and Trial-Size Scenarios";
run;

                               Exemplary Cell Means: Conjectured Delivery Delay (minutes)                               

The GLMPOWER Procedure


      Fixed Scenario Elements       

Item                Value           
------------------  ----------------
Dependent Variable  mean_delay_min  
Source              ROUTING_STRATEGY
Source              DEPOT_REGION    

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Power Curve for the Enrollment Decision

Finally, we trace a **power curve** -- achieved power for the omnibus strategy effect as total enrollment grows from 30 to 270 drivers in steps of 30 -- using the strategy-plus-region model at the expected 8-minute standard deviation. Solving `POWER = .` across that enrollment grid produces the curve as a tabulated `N Total` vs. `Power` series, so we can read off the enrollment where each of the conventional 80% and 90% targets is crossed.

In [4]:
/* ================================================================
   Power curve: achieved power vs. total drivers enrolled, swept
   from 30 to 270 in steps of 30 at the expected 8-min std dev.
   ================================================================ */
proc glmpower data=routing_trial;
   class routing_strategy depot_region;
   model mean_delay_min = routing_strategy depot_region;
   weight cell_n;
   power
      nfractional
      stddev = 8.0
      alpha  = 0.05
      ntotal = 30 60 90 120 150 180 210 240 270
      power  = .;
   title "Power Curve: Achieved Power vs. Total Drivers Enrolled";
run;

                               Exemplary Cell Means: Conjectured Delivery Delay (minutes)                               

The GLMPOWER Procedure


      Fixed Scenario Elements       

Item                Value           
------------------  ----------------
Dependent Variable  mean_delay_min  
Source              ROUTING_STRATEGY
Source              DEPOT_REGION    

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretation and Recommendation

The analysis gives the operations team a defensible enrollment plan:

- **Baseline sizing (Section 2).** Sizing on the headline single-degree-of-freedom contrast -- **AI-Optimized vs. Static** delay -- and assuming an 8-minute residual standard deviation, the trial reaches **80% power at 63 drivers** (achieved power **0.8035**) and **90% power at 83 drivers** (achieved power **0.9014**). Because `NFRACTIONAL` rounds the exact required total *up* to the next whole driver, the realized power sits just above each nominal target rather than landing exactly on it. Rounding further for attrition, an enrollment target near **90 drivers** comfortably clears 90% power for this comparison.

- **Sensitivity matters (Section 3).** Switching the lens to the **omnibus strategy effect** (the 2-degree-of-freedom test across all three strategies), power is highly sensitive to delay variability. At 90 drivers, achieved power falls from **0.9868** (SD=6) to **0.8681** (SD=8) to **0.6782** (SD=10). A 45-driver pilot is adequate only if variability is low (**0.8084** at SD=6) and is badly underpowered (**0.3729**) if SD reaches 10. The practical implication: investing in consistent, well-instrumented routes to hold variability down is as valuable as adding drivers.

- **The power curve (Section 4).** Traced for the same omnibus strategy effect at SD=8, achieved power climbs from **0.3733** at 30 drivers to **0.6887** at 60, **0.8681** at 90, and **0.9500** at 120, flattening above 0.99 by 180. Reading the curve against the targets, 80% power for the omnibus test arrives near **77 drivers** and 90% near **99** -- higher than the 63/83 sizing in Section 2 because the omnibus 2-df strategy test is more demanding than the single 1-df AI-Opt-vs-Static contrast.

**Recommendation:** Enroll approximately **90 drivers** (30 per routing strategy, balanced across the three depot regions). At the expected 8-minute standard deviation this clears 90% power for the headline AI-Opt-vs-Static contrast and holds **0.8681** power for the broader omnibus strategy test, while keeping the trial small enough to execute within a single operating quarter.

*Note:* GLMPOWER analyzes the conjectured *design*, not observed outcomes -- so the credibility of these numbers rests on the conjectured means and standard deviation. Teams should revisit the sizing once a brief pilot yields an empirical estimate of delivery-delay variability.